In [ ]:
#Please note - I have changed and tested with various parameters. Also need to update to all the directories and paths for data where needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

ZIP_PATH = ""
EXTRACT_PATH = ""

if not os.path.exists(ZIP_PATH):
    print(f"ZIP file not found at {ZIP_PATH}")
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        
        zip_ref.extractall(EXTRACT_PATH)

    all_files = []
    for root, dirs, files in os.walk(EXTRACT_PATH):
        for file in files:
            all_files.append(os.path.join(root, file))

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    TimeDistributed,
    GlobalAveragePooling2D,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    Input
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
FRAMES_DIR = "" 
IMG_SIZE = 224
NUM_FRAMES = 64
BATCH_SIZE = 8 # we can try differnt parameters

In [ ]:
CSV_FILE = ""
df = pd.read_csv(CSV_FILE)

In [ ]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["engagement_label"])

num_classes = len(label_encoder.classes_)

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [ ]:
from tensorflow.keras.utils import Sequence

class VideoSequenceGenerator(Sequence):
    def __init__(self, dataframe, batch_size, frames_dir):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.frames_dir = frames_dir

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size:(idx + 1) * self.batch_size]
        X, y = [], []

        for _, row in batch_df.iterrows():
            video_folder = os.path.join(self.frames_dir, str(row["video_filename"]))

            try:
                all_frames = sorted(os.listdir(video_folder))
            except (FileNotFoundError, NotADirectoryError):
                continue

            total_available = len(all_frames)
            if total_available >= NUM_FRAMES:
                indices = np.linspace(0, total_available - 1, NUM_FRAMES).astype(int)
                selected_frames = [all_frames[i] for i in indices]
            else:
                selected_frames = all_frames

            video_frames = []
            for frame_name in selected_frames:
                frame_path = os.path.join(video_folder, frame_name)
                img = cv2.imread(frame_path)
                if img is None: continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                img = tf.keras.applications.efficientnet.preprocess_input(img)
                video_frames.append(img)

            while len(video_frames) < NUM_FRAMES:
                video_frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

            X.append(video_frames)
            y.append(row["label"])

        return np.array(X, dtype='float32'), tf.keras.utils.to_categorical(y, num_classes=3)

In [ ]:
LOCAL_DATA_DIR = ""

train_gen = VideoSequenceGenerator(
    dataframe=train_df,
    batch_size=BATCH_SIZE,
    frames_dir=LOCAL_DATA_DIR
)

val_gen = VideoSequenceGenerator(
    dataframe=val_df,
    batch_size=BATCH_SIZE,
    frames_dir=LOCAL_DATA_DIR
)

In [ ]:

import tensorflow as tf
from tensorflow.keras.layers import Input, TimeDistributed, GlobalAveragePooling2D, Bidirectional, LSTM, Dropout, Dense, SpatialDropout3D
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Model

def build_efficientnet_bilstm(input_shape=(64, 224, 224, 3), num_classes=3):
    base_cnn = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape[1:])

    base_cnn.trainable = True
    for layer in base_cnn.layers[:-50]:
        layer.trainable = False

    video_input = Input(shape=input_shape)

    x = SpatialDropout3D(0.2)(video_input)

    x = TimeDistributed(base_cnn)(x)
    x = TimeDistributed(tf.keras.layers.GlobalMaxPooling2D())(x)

 
    x = Bidirectional(LSTM(128,
                           return_sequences=False,
                           kernel_regularizer=l2(1e-4),
                           recurrent_regularizer=l2(1e-4)))(x)

    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.4)(x)

    outputs = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(video_input, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

tf.keras.backend.clear_session()
eff_model = build_efficientnet_bilstm()
eff_model.summary()

In [ ]:
print(eff_model.input_shape)

In [ ]:

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,             
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,              
        patience=5,              
        min_lr=1e-8,             
        verbose=1
    ),


    tf.keras.callbacks.ModelCheckpoint(
        'best_effnet_32f.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

In [ ]:
from sklearn.utils import class_weight


y_train = train_df["label"].values

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

ce_dict = dict(enumerate(weights))

print(f"Mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")
print(f"Class Weights: {ce_dict}")

In [ ]:
import gc

tf.keras.backend.clear_session()
gc.collect()

eff_model = build_efficientnet_bilstm()

In [ ]:
history = eff_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    class_weight=cw_dict,
    callbacks=callbacks,
    verbose=1  
)

In [ ]:
preds = eff_model.predict(val_gen, verbose=0)

y_pred = np.argmax(preds, axis=1)
y_true = val_df["label"].values

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_
))

In [ ]:
eff_model.save('efficientnet_model.h5')